# Model Evaluation - Nexus-LLM

This notebook covers how to evaluate LLM performance using standard benchmarks and custom metrics.

## Evaluation Approaches

1. **Standard benchmarks** - MMLU, HumanEval, GSM8K, etc.
2. **Reference-based metrics** - ROUGE, BLEU, BERTScore
3. **LLM-as-judge** - Use a stronger model to evaluate outputs
4. **Custom metrics** - Domain-specific evaluation criteria

## 1. Setup

In [ ]:
from nexus_llm import InferenceEngine
from nexus_llm.evaluation import Evaluator, EvalConfig, BenchmarkDataset

engine = InferenceEngine(model_name="nexus-7b-chat", device="auto")

eval_config = EvalConfig(
    batch_size=8,
    temperature=0.0,         # Deterministic for evaluation
    max_tokens=256,
    seed=42,
    output_dir="./eval_results",
)

evaluator = Evaluator(engine=engine, config=eval_config)

## 2. Standard Benchmark: MMLU

In [ ]:
# Evaluate on MMLU (Massive Multitask Language Understanding)
mmlu_results = evaluator.evaluate_benchmark(
    benchmark=BenchmarkDataset.MMLU,
    subsets=["stem", "humanities", "social_sciences", "other"],
)

print(f"MMLU Overall: {mmlu_results.overall_score:.3f}")
print(f"\nSubset Breakdown:")
for subset, score in sorted(mmlu_results.subset_scores.items()):
    print(f"  {subset}: {score:.3f}")

In [ ]:
# Visualize MMLU results
import matplotlib.pyplot as plt

subsets = list(mmlu_results.subset_scores.keys())
scores = list(mmlu_results.subset_scores.values())

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(subsets, scores, color='steelblue')
ax.set_ylabel('Accuracy')
ax.set_title('MMLU Benchmark Results by Subject Area')
ax.set_ylim(0, 1.0)

for bar, score in zip(bars, scores):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
            f'{score:.2f}', ha='center', va='bottom')

plt.tight_layout()
plt.show()

## 3. Code Generation: HumanEval

In [ ]:
# Evaluate code generation
humaneval_results = evaluator.evaluate_benchmark(
    benchmark=BenchmarkDataset.HUMAN_EVAL,
    metric="pass@k",
    k_values=[1, 10, 100],
)

print("HumanEval Code Generation Results:")
for k, score in humaneval_results.scores.items():
    print(f"  pass@{k}: {score:.3f}")

# Show an example
example = humaneval_results.examples[0]
print(f"\nExample prompt:")
print(example.prompt[:200])
print(f"\nGenerated code:")
print(example.completion[:200])
print(f"\nPassed: {example.passed}")

## 4. Custom Evaluation with Reference Metrics

In [ ]:
# Evaluate on a custom QA dataset
custom_results = evaluator.evaluate_custom(
    dataset_path="data/eval_qa_pairs.jsonl",
    input_field="question",
    reference_field="answer",
    metrics=["rouge_1", "rouge_2", "rouge_l", "bleu", "bertscore", "exact_match"],
)

print("Custom QA Evaluation Results:")
for metric, score in custom_results.metric_scores.items():
    print(f"  {metric}: {score:.4f}")

## 5. LLM-as-Judge Evaluation

In [ ]:
from nexus_llm.evaluation import LLMBasedJudge

# Use a stronger model as judge
judge = LLMBasedJudge(
    model_name="nexus-13b-chat",
    criteria=[
        "relevance: Does the response address the question?",
        "accuracy: Is the information factually correct?",
        "completeness: Does it cover all aspects of the question?",
        "clarity: Is the response well-organized and easy to understand?",
    ],
    scale=10,                  # Score from 1-10
)

# Evaluate responses
evaluations = judge.evaluate_batch(
    questions=["Explain quantum computing", "What is machine learning?"],
    responses=[
        "Quantum computing uses quantum bits (qubits) that can exist in superposition...",
        "Machine learning is a subset of AI where computers learn from data...",
    ],
)

for eval_result in evaluations:
    print(f"Question: {eval_result.question}")
    print(f"Overall score: {eval_result.overall_score:.1f}/10")
    for criterion, score in eval_result.criterion_scores.items():
        print(f"  {criterion}: {score:.1f}/10")
    print()

## 6. Model Comparison

In [ ]:
models = ["nexus-3b-chat", "nexus-7b-chat", "nexus-13b-chat"]
comparison = {}

for model_name in models:
    engine = InferenceEngine(model_name=model_name, device="auto")
    evaluator = Evaluator(engine=engine, config=eval_config)
    results = evaluator.evaluate_benchmark(BenchmarkDataset.MMLU)
    comparison[model_name] = results

# Create comparison table
print(f"{'Model':<20} {'MMLU':<10} {'STEM':<10} {'Humanities':<12} {'Social Sci':<12}")
print("-" * 64)
for model_name, results in comparison.items():
    print(f"{model_name:<20} "
          f"{results.overall_score:<10.3f} "
          f"{results.subset_scores.get('stem', 0):<10.3f} "
          f"{results.subset_scores.get('humanities', 0):<12.3f} "
          f"{results.subset_scores.get('social_sciences', 0):<12.3f}")